# D16 — Análise descritiva da amostra causal

Esta etapa descreve a amostra materializada no D15 **antes de qualquer estimação causal**. As comparações entre tratados e controles são descritivas: não estimam ATT, não testam efeitos, não executam event study causal, Callaway–Sant'Anna, matching ou seleção por resultado.

A unidade é município-ano, a janela é 2007–2019 e o outcome primário congelado é `pessoal_ocupado_assalariado` do CEMPRE. O objetivo é documentar composição, cobertura, escala, assimetria e suporte observacional para os diagnósticos pré-estimação.

In [1]:
from pathlib import Path
import hashlib
import importlib.util
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import analisa_amostra_causal as d16
from visualizacao_ipt import CORES_IPT, aplicar_tema_ipt, estilizar_tabela_ipt, salvar_figura_ipt

AMOSTRA_PATH = ROOT / 'data' / 'processed' / 'amostra_causal_cempre_2007_2019.parquet'
PAINEL_INTEGRADO_PATH = ROOT / 'data' / 'processed' / 'cempre_painel_analitico_com_cadastro_causal_2007_2019.parquet'
DIAGNOSTICS = ROOT / 'outputs' / 'diagnostics'
FIGURES = ROOT / 'outputs' / 'figures'

amostra = pd.read_parquet(AMOSTRA_PATH)
painel_integrado = pd.read_parquet(PAINEL_INTEGRADO_PATH)
validacao = d16.validar_amostra_causal(amostra)

def sha256(caminho: Path) -> str:
    h = hashlib.sha256()
    with caminho.open('rb') as arquivo:
        for bloco in iter(lambda: arquivo.read(1 << 20), b''):
            h.update(bloco)
    return h.hexdigest()

provenance = pd.DataFrame([{
    'artefato': AMOSTRA_PATH.name,
    'caminho_relativo': AMOSTRA_PATH.relative_to(ROOT).as_posix(),
    'bytes': AMOSTRA_PATH.stat().st_size,
    'sha256': sha256(AMOSTRA_PATH),
    'granularidade': 'municipio-ano',
    'chave': 'codigo_municipio_ibge + ano',
}])
display(estilizar_tabela_ipt(provenance))
print('Validação estrutural D14/D15: OK')

,artefato,caminho_relativo,bytes,sha256,granularidade,chave
0,amostra_causal_cempre_2007_2019.parquet,data/processed/amostra_causal_cempre_2007_2019.parquet,1824285,7c24c01b569ed6d10f773d00599b5010e1607696f46d716827182a078e4d733b,municipio-ano,codigo_municipio_ibge + ano


Validação estrutural D14/D15: OK


## 1. Dimensão e composição

O painel-base preserva os 13 anos para cada unidade. A máscara de elegibilidade atua sobre uma observação, sem remover Cabo Frio da população nem desbalancear o artefato-base.

In [2]:
resumo_amostra = d16.resumo_amostra_causal(amostra)
composicao = (amostra.groupby('papel_causal', as_index=False)
              .agg(n_municipios=('codigo_municipio_ibge', 'nunique'), n_observacoes=('ano', 'size')))
display(estilizar_tabela_ipt(resumo_amostra), estilizar_tabela_ipt(composicao))

fig = go.Figure(go.Bar(
    x=composicao['papel_causal'], y=composicao['n_municipios'],
    marker_color=[CORES_IPT['AZUL_CLARO'], CORES_IPT['AZUL_PRINCIPAL']],
    text=composicao['n_municipios'], textposition='outside',
    hovertemplate='%{x}<br>%{y:,.0f} municípios<extra></extra>',
))
aplicar_tema_ipt(fig, titulo='Composição da amostra causal por papel')
fig.update_xaxes(title='')
fig.update_yaxes(title='Municípios', rangemode='tozero')
destino_composicao = salvar_figura_ipt(fig, FIGURES / '11_composicao_amostra_d16.png')
fig.show()

,medida,valor
0,tratados,129
1,controles_never_treated,4963
2,municipios_total,5092
3,municipio_anos_painel_base,66196
4,observacoes_elegiveis_estimacao,66195
5,observacoes_inelegiveis_estimacao,1
6,anos_inicio,2007
7,anos_fim,2019


,papel_causal,n_municipios,n_observacoes
0,CONTROLE_NEVER_TREATED,4963,64519
1,TRATADO_PRINCIPAL,129,1677


## 2. Coortes, distribuição territorial e cobertura temporal

As coortes são contadas por município, não por linha. Região e UF são descritas somente quando presentes no artefato. O código da macrorregião deriva do primeiro dígito do código da UF do IBGE e serve apenas para agregação descritiva.

In [3]:
distribuicao_coortes = d16.distribuicao_coortes(amostra)
mapa_regiao = {'1': 'Norte', '2': 'Nordeste', '3': 'Sudeste', '4': 'Sul', '5': 'Centro-Oeste'}
municipios = amostra.sort_values('ano').drop_duplicates('codigo_municipio_ibge').copy()
municipios['regiao'] = municipios['uf_codigo'].astype(str).str[0].map(mapa_regiao)
por_regiao = (municipios.groupby(['regiao', 'papel_causal'], as_index=False, dropna=False)
              .agg(n_municipios=('codigo_municipio_ibge', 'nunique')))
por_uf = (municipios.groupby(['uf_codigo', 'papel_causal'], as_index=False, dropna=False)
          .agg(n_municipios=('codigo_municipio_ibge', 'nunique')))
cobertura_ano = (amostra.groupby(['ano', 'papel_causal'], as_index=False)
                 .agg(n_municipios=('codigo_municipio_ibge', 'nunique'),
                      n_observacoes=('codigo_municipio_ibge', 'size'),
                      n_elegiveis=('elegivel_estimacao_principal', 'sum')))
display(estilizar_tabela_ipt(distribuicao_coortes), estilizar_tabela_ipt(por_regiao), estilizar_tabela_ipt(por_uf), estilizar_tabela_ipt(cobertura_ano))

fig = go.Figure(go.Bar(
    x=distribuicao_coortes['coorte_g'].astype(str), y=distribuicao_coortes['n_tratados'],
    marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=distribuicao_coortes['n_tratados'], textposition='outside',
    hovertemplate='Coorte %{x}<br>%{y} tratados<extra></extra>',
))
aplicar_tema_ipt(fig, titulo='Distribuição dos tratados por coorte')
fig.update_xaxes(title='Coorte', type='category')
fig.update_yaxes(title='Municípios tratados', rangemode='tozero')
destino_coortes = salvar_figura_ipt(fig, FIGURES / '12_distribuicao_coortes_d16.png')
fig.show()

,coorte_g,n_tratados,n_observacoes_painel,n_observacoes_elegiveis
0,2009,21,273,273
1,2010,27,351,350
2,2011,66,858,858
3,2012,13,169,169
4,2013,2,26,26


,regiao,papel_causal,n_municipios
0,Centro-Oeste,CONTROLE_NEVER_TREATED,410
1,Centro-Oeste,TRATADO_PRINCIPAL,17
2,Nordeste,CONTROLE_NEVER_TREATED,1578
3,Nordeste,TRATADO_PRINCIPAL,41
4,Norte,CONTROLE_NEVER_TREATED,385
5,Norte,TRATADO_PRINCIPAL,18
6,Sudeste,CONTROLE_NEVER_TREATED,1518
7,Sudeste,TRATADO_PRINCIPAL,31
8,Sul,CONTROLE_NEVER_TREATED,1072
9,Sul,TRATADO_PRINCIPAL,22


,uf_codigo,papel_causal,n_municipios
0,11,CONTROLE_NEVER_TREATED,44
1,11,TRATADO_PRINCIPAL,2
2,12,CONTROLE_NEVER_TREATED,17
3,12,TRATADO_PRINCIPAL,2
4,13,CONTROLE_NEVER_TREATED,49
5,13,TRATADO_PRINCIPAL,5
6,14,CONTROLE_NEVER_TREATED,11
7,14,TRATADO_PRINCIPAL,1
8,15,CONTROLE_NEVER_TREATED,126
9,15,TRATADO_PRINCIPAL,4


,ano,papel_causal,n_municipios,n_observacoes,n_elegiveis
0,2007,CONTROLE_NEVER_TREATED,4963,4963,4963
1,2007,TRATADO_PRINCIPAL,129,129,129
2,2008,CONTROLE_NEVER_TREATED,4963,4963,4963
3,2008,TRATADO_PRINCIPAL,129,129,129
4,2009,CONTROLE_NEVER_TREATED,4963,4963,4963
5,2009,TRATADO_PRINCIPAL,129,129,128
6,2010,CONTROLE_NEVER_TREATED,4963,4963,4963
7,2010,TRATADO_PRINCIPAL,129,129,129
8,2011,CONTROLE_NEVER_TREATED,4963,4963,4963
9,2011,TRATADO_PRINCIPAL,129,129,129


## 3. Cobertura e status do outcome; casos especiais

A taxonomia real do outcome é preservada. `zero_real` é valor observado igual a zero, não ausência. O sigilo que motivou a exclusão da unidade `5003900` não deve reaparecer no painel final.

In [4]:
status_outcome = (amostra.groupby(['papel_causal', 'status_pessoal_ocupado_assalariado'], as_index=False, dropna=False)
                  .agg(n_observacoes=('codigo_municipio_ibge', 'size'),
                       n_outcome_ausente=('pessoal_ocupado_assalariado', lambda s: int(s.isna().sum()))))
cabo_frio_2009 = amostra.loc[(amostra.codigo_municipio_ibge.astype(str).eq('3300704')) & amostra.ano.eq(2009),
                               ['codigo_municipio_ibge', 'ano', 'coorte_g', 'tempo_relativo', 'pessoal_ocupado_assalariado',
                                'status_pessoal_ocupado_assalariado', 'elegivel_estimacao_principal',
                                'motivo_nao_elegibilidade_estimacao']]
verificacao_especiais = pd.DataFrame([
    {'verificacao': 'Cabo Frio/2009 existe no painel-base', 'valor': len(cabo_frio_2009), 'esperado': 1},
    {'verificacao': 'Cabo Frio/2009 elegivel para estimacao', 'valor': int(cabo_frio_2009.elegivel_estimacao_principal.sum()), 'esperado': 0},
    {'verificacao': 'linhas do codigo 5003900', 'valor': int(amostra.codigo_municipio_ibge.astype(str).eq('5003900').sum()), 'esperado': 0},
    {'verificacao': 'NaNs no outcome final', 'valor': int(amostra.pessoal_ocupado_assalariado.isna().sum()), 'esperado': 0},
    {'verificacao': 'status sigilo no outcome final', 'valor': int(amostra.status_pessoal_ocupado_assalariado.eq('sigilo').sum()), 'esperado': 0},
])
display(estilizar_tabela_ipt(status_outcome), estilizar_tabela_ipt(cabo_frio_2009), estilizar_tabela_ipt(verificacao_especiais))

,papel_causal,status_pessoal_ocupado_assalariado,n_observacoes,n_outcome_ausente
0,CONTROLE_NEVER_TREATED,observado,64516,0
1,CONTROLE_NEVER_TREATED,zero_real,3,0
2,TRATADO_PRINCIPAL,observado,1677,0


,codigo_municipio_ibge,ano,coorte_g,tempo_relativo,pessoal_ocupado_assalariado,status_pessoal_ocupado_assalariado,elegivel_estimacao_principal,motivo_nao_elegibilidade_estimacao
37585,3300704,2009,2010.000000,-1.000000,33607.000000,observado,False,ANO_TRANSICAO_INSTITUCIONAL_CABO_FRIO_2009


,verificacao,valor,esperado
0,Cabo Frio/2009 existe no painel-base,1,1
1,Cabo Frio/2009 elegivel para estimacao,0,0
2,linhas do codigo 5003900,0,0
3,NaNs no outcome final,0,0
4,status sigilo no outcome final,0,0


## 4. Estatísticas, quantis, escala e assimetria do outcome

As diferenças de nível entre os grupos refletem composição e escala municipal; não são efeitos. A distribuição em nível é fortemente assimétrica, por isso a visualização usa `log1p` apenas como recurso descritivo. O outcome congelado permanece em nível.

In [5]:
def estatisticas(frame: pd.DataFrame, grupo: str) -> dict:
    s = frame['pessoal_ocupado_assalariado']
    return {
        'grupo': grupo, 'n': int(s.count()), 'media': s.mean(), 'desvio_padrao': s.std(),
        'min': s.min(), 'p01': s.quantile(.01), 'p05': s.quantile(.05),
        'p25': s.quantile(.25), 'mediana': s.median(), 'p75': s.quantile(.75),
        'p95': s.quantile(.95), 'p99': s.quantile(.99), 'max': s.max(), 'assimetria': s.skew(),
    }

estatisticas_outcome = pd.DataFrame([estatisticas(amostra, 'TOTAL')] + [
    estatisticas(grupo, papel) for papel, grupo in amostra.groupby('papel_causal')
])
estatisticas_coorte = (amostra.loc[amostra.papel_causal.eq('TRATADO_PRINCIPAL')]
                       .groupby('coorte_g', as_index=False)
                       .agg(n=('pessoal_ocupado_assalariado', 'count'),
                            media=('pessoal_ocupado_assalariado', 'mean'),
                            mediana=('pessoal_ocupado_assalariado', 'median'),
                            p25=('pessoal_ocupado_assalariado', lambda s: s.quantile(.25)),
                            p75=('pessoal_ocupado_assalariado', lambda s: s.quantile(.75))))
display(estilizar_tabela_ipt(estatisticas_outcome), estilizar_tabela_ipt(estatisticas_coorte))

fig = go.Figure()
for papel, cor in [('CONTROLE_NEVER_TREATED', CORES_IPT['AZUL_CLARO']), ('TRATADO_PRINCIPAL', CORES_IPT['AZUL_PRINCIPAL'])]:
    valores = np.log1p(amostra.loc[amostra.papel_causal.eq(papel), 'pessoal_ocupado_assalariado'])
    fig.add_trace(go.Histogram(x=valores, name=papel, histnorm='probability density', opacity=.65, nbinsx=60, marker_color=cor))
fig.update_layout(barmode='overlay')
aplicar_tema_ipt(fig, titulo='Distribuição descritiva do outcome em escala log1p')
fig.update_xaxes(title='log(1 + pessoal ocupado assalariado)')
fig.update_yaxes(title='Densidade')
destino_distribuicao = salvar_figura_ipt(fig, FIGURES / '13_distribuicao_outcome_log1p_d16.png')
fig.show()

,grupo,n,media,desvio_padrao,min,p01,p05,p25,mediana,p75,p95,p99,max,assimetria
0,TOTAL,66196,3158.668333,11108.334884,0.000000,127.000000,200.000000,398.000000,813.000000,2113.000000,11425.250000,42690.850000,292385.000000,12.395531
1,CONTROLE_NEVER_TREATED,64519,2729.514066,10182.554179,0.000000,127.000000,199.000000,391.000000,783.000000,1942.000000,8997.400000,34940.120000,292385.000000,14.806543
2,TRATADO_PRINCIPAL,1677,19669.463924,24543.378820,176.000000,385.600000,1411.800000,4419.000000,11024.000000,24747.000000,70965.000000,115674.640000,179317.000000,2.715011


,coorte_g,n,media,mediana,p25,p75
0,2009.000000,273,24045.934066,18582.000000,6763.000000,35941.000000
1,2010.000000,351,14384.159544,8409.000000,3227.500000,21812.000000
2,2011.000000,858,18167.743590,9898.000000,3974.000000,18359.250000
3,2012.000000,169,30566.674556,20943.000000,8610.000000,42031.000000
4,2013.000000,26,23793.038462,20004.500000,11906.500000,33815.500000


## 5. Evolução temporal descritiva

As linhas abaixo mostram medianas anuais dentro de grupos cuja composição foi definida ex ante. Divergências de trajetória não identificam impacto: podem refletir seleção, escala, choques e outras diferenças entre municípios.

In [6]:
evolucao_ano = (amostra.groupby(['ano', 'papel_causal'], as_index=False)
                .agg(media=('pessoal_ocupado_assalariado', 'mean'),
                     mediana=('pessoal_ocupado_assalariado', 'median'),
                     p25=('pessoal_ocupado_assalariado', lambda s: s.quantile(.25)),
                     p75=('pessoal_ocupado_assalariado', lambda s: s.quantile(.75))))
display(estilizar_tabela_ipt(evolucao_ano))

fig = go.Figure()
for papel, cor in [('CONTROLE_NEVER_TREATED', CORES_IPT['AZUL_CLARO']), ('TRATADO_PRINCIPAL', CORES_IPT['AZUL_PRINCIPAL'])]:
    serie = evolucao_ano.loc[evolucao_ano.papel_causal.eq(papel)]
    fig.add_trace(go.Scatter(x=serie.ano, y=serie.mediana, mode='lines+markers', name=papel, line={'color': cor},
                             customdata=serie[['media', 'p25', 'p75']],
                             hovertemplate='Ano %{x}<br>Mediana %{y:,.0f}<br>Média %{customdata[0]:,.0f}<br>P25–P75 %{customdata[1]:,.0f}–%{customdata[2]:,.0f}<extra></extra>'))
aplicar_tema_ipt(fig, titulo='Evolução descritiva da mediana do outcome')
fig.update_xaxes(title='Ano', dtick=1)
fig.update_yaxes(title='Pessoal ocupado assalariado (mediana)')
destino_evolucao = salvar_figura_ipt(fig, FIGURES / '14_evolucao_outcome_d16.png')
fig.show()

,ano,papel_causal,media,mediana,p25,p75
0,2007,CONTROLE_NEVER_TREATED,2246.382430,638.000000,309.000000,1547.500000
1,2007,TRATADO_PRINCIPAL,15744.674419,8182.000000,3316.000000,19086.000000
2,2008,CONTROLE_NEVER_TREATED,2322.327826,644.000000,315.000000,1612.500000
3,2008,TRATADO_PRINCIPAL,16462.961240,8924.000000,3407.000000,20379.000000
4,2009,CONTROLE_NEVER_TREATED,2449.761032,702.000000,350.000000,1720.500000
5,2009,TRATADO_PRINCIPAL,17375.527132,9464.000000,3680.000000,21622.000000
6,2010,CONTROLE_NEVER_TREATED,2642.260729,746.000000,367.500000,1832.000000
7,2010,TRATADO_PRINCIPAL,18687.418605,10318.000000,4335.000000,22692.000000
8,2011,CONTROLE_NEVER_TREATED,2764.240580,789.000000,390.000000,1929.000000
9,2011,TRATADO_PRINCIPAL,19896.116279,11143.000000,4403.000000,24915.000000


## 6. Event-time como cobertura, sem ATT

`tempo_relativo = ano - coorte_g` é usado aqui somente para contar observações disponíveis. Há duas quantidades distintas:

1. **população com suporte de calendário**: 129 na janela principal e 108 na sensibilidade (coortes 2010–2013);
2. **unidades com todos os k elegíveis** após a máscara institucional: Cabo Frio/2009 corresponde a `k=-1`, reduzindo a cobertura completa para 128 e 107, respectivamente.

Essa perda de uma unidade não altera a população congelada; documenta uma restrição observacional que o futuro estimador deverá respeitar.

In [7]:
cobertura_event_time = d16.cobertura_event_time(amostra)
diagnostico_janelas = d16.diagnostico_janelas(amostra)
display(estilizar_tabela_ipt(cobertura_event_time), estilizar_tabela_ipt(diagnostico_janelas))

fig = go.Figure()
fig.add_trace(go.Scatter(x=cobertura_event_time.tempo_relativo, y=cobertura_event_time.n_tratados_no_painel,
                         mode='lines+markers', name='No painel-base', line={'color': CORES_IPT['AZUL_CLARO']}))
fig.add_trace(go.Scatter(x=cobertura_event_time.tempo_relativo, y=cobertura_event_time.n_tratados_elegiveis,
                         mode='lines+markers', name='Elegíveis', line={'color': CORES_IPT['AZUL_PRINCIPAL']}))
aplicar_tema_ipt(fig, titulo='Cobertura descritiva dos tratados por event-time')
fig.update_xaxes(title='k = ano - coorte', dtick=1)
fig.update_yaxes(title='Municípios tratados', rangemode='tozero')
fig.add_vrect(x0=-2.5, x1=2.5, fillcolor=CORES_IPT['CIANO'], opacity=.08, line_width=0, annotation_text='janela principal')
destino_event_time = salvar_figura_ipt(fig, FIGURES / '15_cobertura_event_time_d16.png')
fig.show()

,tempo_relativo,n_tratados_no_painel,n_tratados_elegiveis,perda_por_mascara
0,-6,2,2,0
1,-5,15,15,0
2,-4,81,81,0
3,-3,108,108,0
4,-2,129,129,0
5,-1,129,128,1
6,0,129,129,0
7,1,129,129,0
8,2,129,129,0
9,3,129,129,0


,janela,k_min,k_max,n_periodos,n_tratados_populacao,n_tratados_janela_completa_elegivel,perda_por_cobertura
0,principal_k_-2_2,-2,2,5,129,128,1
1,sensibilidade_k_-3_2,-3,2,6,108,107,1


## 7. Funil da população e artefatos reutilizáveis

O funil visual usa somente **unidades municipais**, evitando comparar municípios com linhas. A máscara de Cabo Frio é mostrada separadamente na tabela de observações: ela reduz 66.196 para 66.195 linhas elegíveis, sem excluir município.

In [8]:
n_universo = painel_integrado.codigo_municipio_ibge.nunique()
n_tratados = validacao['n_tratados']
n_pool = painel_integrado.loc[painel_integrado.fl_elegivel_controle_candidato, 'codigo_municipio_ibge'].nunique()
funil_unidades = pd.DataFrame({
    'etapa': ['Universo integrado', 'Tratados + pool estrutural', 'Após exclusão do sigilo'],
    'municipios': [n_universo, n_tratados + n_pool, validacao['n_unidades']],
})
funil_observacoes = pd.DataFrame({
    'etapa': ['Painel-base balanceado', 'Após máscara Cabo Frio/2009'],
    'observacoes': [validacao['n_linhas'], validacao['n_linhas_elegiveis']],
})
display(estilizar_tabela_ipt(funil_unidades), estilizar_tabela_ipt(funil_observacoes))

fig = go.Figure(go.Funnel(
    y=funil_unidades['etapa'], x=funil_unidades['municipios'],
    textinfo='value+percent initial', marker={'color': [CORES_IPT['AZUL_CLARO'], CORES_IPT['CIANO'], CORES_IPT['AZUL_PRINCIPAL']]},
))
aplicar_tema_ipt(fig, titulo='Funil de unidades da amostra causal')
fig.add_annotation(text='A máscara observacional não reduz unidades: 66.196 → 66.195 linhas elegíveis',
                   xref='paper', yref='paper', x=0, y=-.18, showarrow=False, xanchor='left')
destino_funil = salvar_figura_ipt(fig, FIGURES / '16_funil_amostra_d16.png')
fig.show()

status_metricas = pd.DataFrame([
    {'medida': f'status_outcome_{status}', 'valor': int(n)}
    for status, n in amostra.status_pessoal_ocupado_assalariado.value_counts(dropna=False).items()
])
janela_metricas = pd.DataFrame([
    {'medida': f"{row.janela}_populacao", 'valor': int(row.n_tratados_populacao)}
    for row in diagnostico_janelas.itertuples(index=False)
] + [
    {'medida': f"{row.janela}_completa_elegivel", 'valor': int(row.n_tratados_janela_completa_elegivel)}
    for row in diagnostico_janelas.itertuples(index=False)
])
resumo_exportacao = pd.concat([resumo_amostra, status_metricas, janela_metricas], ignore_index=True)
resumo_exportacao.to_csv(DIAGNOSTICS / 'resumo_amostra_causal.csv', index=False, encoding='utf-8')
distribuicao_coortes.to_csv(DIAGNOSTICS / 'distribuicao_coortes.csv', index=False, encoding='utf-8')
cobertura_event_time.to_csv(DIAGNOSTICS / 'cobertura_event_time.csv', index=False, encoding='utf-8')
print('Diagnósticos exportados:', 'resumo_amostra_causal.csv, distribuicao_coortes.csv, cobertura_event_time.csv')

,etapa,municipios
0,Universo integrado,5570
1,Tratados + pool estrutural,5093
2,Após exclusão do sigilo,5092


,etapa,observacoes
0,Painel-base balanceado,66196
1,Após máscara Cabo Frio/2009,66195


Diagnósticos exportados: resumo_amostra_causal.csv, distribuicao_coortes.csv, cobertura_event_time.csv


## 8. Exportação visual

O projeto sempre tenta PNG e preserva HTML interativo como fallback. A célula abaixo diagnostica o ambiente sem instalar dependências.

In [9]:
kaleido_disponivel = importlib.util.find_spec('kaleido') is not None
destinos_figuras = [destino_composicao, destino_coortes, destino_distribuicao, destino_evolucao, destino_event_time, destino_funil]
print(f'Kaleido disponível: {kaleido_disponivel}')
print('Destinos efetivos das figuras:')
for destino in destinos_figuras:
    print(' -', destino.relative_to(ROOT))
if not kaleido_disponivel:
    print('PNG indisponível: falta a dependência kaleido>=1 solicitada pelo Plotly 7.0.0; nenhuma instalação foi realizada. HTML interativo preservado.')

Kaleido disponível: False
Destinos efetivos das figuras:
 - outputs\figures\interactive\11_composicao_amostra_d16.html
 - outputs\figures\interactive\12_distribuicao_coortes_d16.html
 - outputs\figures\interactive\13_distribuicao_outcome_log1p_d16.html
 - outputs\figures\interactive\14_evolucao_outcome_d16.html
 - outputs\figures\interactive\15_cobertura_event_time_d16.html
 - outputs\figures\interactive\16_funil_amostra_d16.html
PNG indisponível: falta a dependência kaleido>=1 solicitada pelo Plotly 7.0.0; nenhuma instalação foi realizada. HTML interativo preservado.


## 9. Síntese e gate D16

A amostra D15 foi reproduzida sem inconsistência bloqueadora. O outcome final tem cobertura completa: 66.193 observações com status `observado` e 3 `zero_real`, sem NaNs nem sigilo. As distribuições de tratados e controles diferem fortemente em escala; isso reforça a necessidade de diagnósticos pré-estimação, mas não constitui efeito causal.

A distinção entre população de janela (129/108) e cobertura completa elegível (128/107) é uma ressalva observacional explícita, não uma alteração de tratamento, timing ou população congelada. Nenhum desenho causal é aprovado nesta etapa.

In [10]:
bloqueadores = []
if validacao != {'n_tratados': 129, 'n_controles': 4963, 'n_unidades': 5092, 'n_linhas': 66196,
                  'n_linhas_elegiveis': 66195, 'sobreposicao_tratado_controle': 0,
                  'unidades_painel_desbalanceado': 0}:
    bloqueadores.append('Dimensões/contratos D15 divergentes.')
if amostra.pessoal_ocupado_assalariado.isna().any():
    bloqueadores.append('Outcome ausente na amostra final.')
if amostra.status_pessoal_ocupado_assalariado.eq('sigilo').any():
    bloqueadores.append('Sigilo reapareceu na amostra final.')

D16_AMOSTRA_DESCRITA = 'SIM'
D16_INCONSISTENCIA_BLOQUEADORA = 'SIM' if bloqueadores else 'NAO'
D16_PRONTA_PARA_DIAGNOSTICOS_PRE_ESTIMACAO = 'NAO' if bloqueadores else 'SIM'
DESENHO_CAUSAL_APROVADO = 'NAO'
print(f'D16_AMOSTRA_DESCRITA = {D16_AMOSTRA_DESCRITA}')
print(f'D16_INCONSISTENCIA_BLOQUEADORA = {D16_INCONSISTENCIA_BLOQUEADORA}')
print(f'D16_PRONTA_PARA_DIAGNOSTICOS_PRE_ESTIMACAO = {D16_PRONTA_PARA_DIAGNOSTICOS_PRE_ESTIMACAO}')
print(f'DESENHO_CAUSAL_APROVADO = {DESENHO_CAUSAL_APROVADO}')
if bloqueadores:
    print('Bloqueadores:', bloqueadores)

D16_AMOSTRA_DESCRITA = SIM
D16_INCONSISTENCIA_BLOQUEADORA = NAO
D16_PRONTA_PARA_DIAGNOSTICOS_PRE_ESTIMACAO = SIM
DESENHO_CAUSAL_APROVADO = NAO
